# SustAInable — Notebook 04: SHAP Explainability
**Project:** SustAInable — Neighborhood Heat Illness Risk Prediction by ZIP Code  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/sustainable-heat](https://github.com/meyeringn/sustainable-heat)

---

## What This Notebook Does

A public health official who receives a list of high-risk ZIP codes needs to know *why* each one was flagged. Is it temperature? Lack of AC access? A high elderly population? The answer determines what intervention to deploy.

This notebook uses **SHAP** (SHapley Additive exPlanations) to make every prediction explainable — globally across the model, and individually for any specific ZIP code.

**Outputs:**
- SHAP beeswarm plot — which features drive risk across all ZIP codes
- SHAP bar plot — ranked mean feature importance
- SHAP dependence plots — how specific features interact with risk
- SHAP waterfall — why one specific high-risk ZIP code was flagged
- `sustainable_shap_values.csv` — full SHAP matrix

In [ ]:
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 1. Load Model and Holdout Data

In [ ]:
model = joblib.load('sustainable_xgboost_model.pkl')
X_holdout = np.load('sustainable_X_holdout.npy')
y_holdout = np.load('sustainable_y_holdout.npy')
feature_cols = joblib.load('sustainable_feature_cols.pkl')

X_holdout_df = pd.DataFrame(X_holdout, columns=feature_cols)
print(f'Holdout: {X_holdout_df.shape}')

## 2. Compute SHAP Values

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_holdout_df)
expected_value = explainer.expected_value

shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df.to_csv('sustainable_shap_values.csv', index=False)
print(f'SHAP values shape: {shap_values.shape}')
print(f'Expected value: {expected_value:.4f}')
print('Saved: sustainable_shap_values.csv')

## 3. Beeswarm Plot — Global Feature Importance

In [ ]:
plt.figure(figsize=(11, 9))
shap.summary_plot(shap_values, X_holdout_df, feature_names=feature_cols, show=False, plot_size=None)
plt.title(
    'SustAInable — SHAP Feature Importance\n'
    'Each dot = one ZIP code. Color = feature value (red=high, blue=low).\n'
    'X-axis position = impact on predicted heat risk.',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('sustainable_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_shap_beeswarm.png')

## 4. Bar Plot — Mean Absolute SHAP

In [ ]:
mean_shap = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=True)

mean_shap.to_csv('sustainable_shap_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(mean_shap['feature'], mean_shap['mean_abs_shap'],
        color='#C62828', alpha=0.85, edgecolor='white')
ax.set_xlabel('Mean |SHAP Value| — Average Impact on Risk Score', fontsize=11)
ax.set_title('SustAInable — Feature Importance\nMean Absolute SHAP Values',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('sustainable_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_shap_bar.png')
print('Saved: sustainable_shap_importance.csv')

## 5. Dependence Plot — AC Access vs Risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SustAInable — SHAP Dependence Plots\nHow individual features drive heat risk predictions',
             fontweight='bold', fontsize=13)

# AC access dependence
ac_idx = feature_cols.index('ac_access_pct')
axes[0].scatter(X_holdout_df['ac_access_pct'], shap_values[:, ac_idx],
                c=X_holdout_df['poverty_rate_pct'], cmap='RdYlGn_r', alpha=0.5, s=20)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('AC Access (%)')
axes[0].set_ylabel('SHAP Value (impact on risk score)')
axes[0].set_title('AC Access — colored by Poverty Rate', fontweight='bold')
sm0 = plt.cm.ScalarMappable(cmap='RdYlGn_r')
sm0.set_array(X_holdout_df['poverty_rate_pct'])
plt.colorbar(sm0, ax=axes[0], label='Poverty Rate (%)')

# Heat hazard index dependence
hhi_idx = feature_cols.index('heat_hazard_index')
axes[1].scatter(X_holdout_df['heat_hazard_index'], shap_values[:, hhi_idx],
                c=X_holdout_df['avg_summer_temp_f'], cmap='hot_r', alpha=0.5, s=20)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Heat Hazard Index (0–10)')
axes[1].set_ylabel('SHAP Value (impact on risk score)')
axes[1].set_title('Heat Hazard Index — colored by Avg Summer Temp', fontweight='bold')
sm1 = plt.cm.ScalarMappable(cmap='hot_r')
sm1.set_array(X_holdout_df['avg_summer_temp_f'])
plt.colorbar(sm1, ax=axes[1], label='Avg Summer Temp (°F)')

plt.tight_layout()
plt.savefig('sustainable_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_shap_dependence.png')

## 6. Waterfall — Why Was This ZIP Code Flagged?

In [ ]:
y_prob = model.predict_proba(X_holdout_df)[:, 1]
highest_risk_idx = np.argmax(y_prob)

print(f'Highest-risk ZIP in holdout:')
print(f'  Risk score: {y_prob[highest_risk_idx]:.4f}')
print(f'  Actual label: {"Elevated Risk" if y_holdout[highest_risk_idx]==1 else "Lower Risk"}')

shap_exp = shap.Explanation(
    values=shap_values[highest_risk_idx],
    base_values=expected_value,
    data=X_holdout_df.iloc[highest_risk_idx].values,
    feature_names=feature_cols
)
plt.figure(figsize=(11, 7))
shap.plots.waterfall(shap_exp, show=False)
plt.title(
    f'SustAInable — Why Was This ZIP Code Flagged?\n'
    f'Risk Score: {y_prob[highest_risk_idx]:.3f} | '
    f'Actual: {"Elevated Risk" if y_holdout[highest_risk_idx]==1 else "Lower Risk"}',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
plt.savefig('sustainable_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sustainable_shap_waterfall.png')

## Key Takeaways

| Finding | Public Health Implication |
|---|---|
| AC access is a top negative driver | AC voucher programs and cooling center proximity are high-leverage interventions |
| Heat Hazard Index tracks closely with SHAP | HHI composite is validated as a useful aggregate signal |
| Social vulnerability index amplifies all heat features | Economic conditions multiply physical heat exposure |
| Waterfall charts are agency-ready | Every flagged ZIP can be explained in plain language to a public health director |

➡️ **Next:** Notebooks 05 & 06 — Threshold Analysis & Agency Report